# 365 Probabilidades - Dia #125
## Qual a probabilidade de a sua rua ter mais cachorros do que gatos?

**Tipo:** Probabilidade simples
**Data de publicação:** 2026-10-16
**Ferramenta:** Python
**Decisão analisada:** A sensação de que rua é lugar de cachorro está certa, e por quanto?
**Hashtag:** #365Probabilidades #Dia125

---

### 📖 A História

Você sabe quais casas da sua rua têm cachorro. Não precisou perguntar. O cachorro se
apresenta no portão, às seis da manhã, quando passa o carteiro, quando passa outro
cachorro.

O gato não. O gato da casa ao lado pode morar ali há dez anos sem que você saiba.

Daí a sensação de que rua é lugar de cachorro. A pergunta de hoje é se a sensação está
certa, e por quanto. Em 2019 o IBGE entrou em 108 mil domicílios e perguntou se havia
cachorro e se havia gato. Com isso dá para montar a sua rua, casa por casa, e contar.

---

### 📚 O Conceito

Uma rua é uma amostra. Cada casa cai num de quatro tipos: tem cachorro e não tem gato,
tem gato e não tem cachorro, tem os dois, não tem nenhum. Numa rua de n casas, a
contagem dos quatro tipos segue uma distribuição multinomial, e a pergunta "mais casas
com cachorro do que com gato" tem resposta exata.

A conta tem um atalho. As casas que têm os dois bichos contam um ponto para cada lado e
se anulam. A disputa fica entre as casas só de cachorro e as casas só de gato. É por
isso que o dado de "os dois" importa, e ele não está publicado de forma direta: sai por
diferença, a partir da tabela de vacinação contra a raiva, que o IBGE calcula sobre os
domicílios com cachorro ou gato.

O IBGE publica cada proporção com intervalo de confiança de 95%. O modelo carrega esses
intervalos até a resposta final, por simulação, para que a incerteza do Censo apareça na
incerteza da rua.

---

### 🧮 O Modelo

Dia de **probabilidade simples**: a pergunta vem primeiro, e o dado é contagem.

**Dados:**
- IBGE, *Pesquisa Nacional de Saúde 2019*. Amostra probabilística de 108.525
  domicílios selecionados (108.457 com entrevista), Brasil, grandes regiões e unidades
  da federação. Tabelas do SIDRA:
  - **4930**, domicílios com algum cachorro: 33.754 mil, **46,1%** [45,5; 46,7];
  - **4931**, domicílios com algum gato: 14.144 mil, **19,3%** [18,9; 19,7];
  - **4932**, domicílios com cachorro ou gato em que todos os animais foram vacinados
    contra a raiva nos 12 meses anteriores: 28.404 mil, **72,0%** [71,4; 72,7] dos
    domicílios com cachorro ou gato. A publicação do IBGE fala em 39,4 milhões de
    domicílios com cachorro ou gato, que é o denominador recuperado aqui.
- As três tabelas, para Brasil (total, urbano e rural) e para as 27 unidades da
  federação, estão em `pns2019_cachorro_gato.csv`, ao lado deste notebook, com os
  intervalos de confiança publicados.

**Modelo próprio:**
- Quatro tipos de domicílio por diferença entre as três tabelas.
- Probabilidade exata, por enumeração da multinomial, de uma rua de n casas ter mais
  casas com cachorro do que com gato (n = 10 como régua; curva de 1 a 40).
- Monte Carlo de 200.000 ruas (semente 42) como conferência da conta exata.
- Propagação dos intervalos de confiança do IBGE até a resposta, por simulação.
- A mesma conta para urbano, rural e cada unidade da federação.
- Sem ×0,80: o dado é um fato da casa (tem ou não tem), não atitude nem comportamento
  lembrado, e o próprio IBGE publica o intervalo. A convenção do projeto vale para
  proporções de survey sobre o que a pessoa diz de si.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from math import lgamma, exp

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
SEMENTE = 42
ger = np.random.default_rng(SEMENTE)

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def pct(x, casas=1):
    return vir(100 * x, casas) + "%"

def mil(n):
    return f"{int(round(n)):,}".replace(",", ".")

print("✅ Bibliotecas carregadas")


In [ ]:
# --- OS DADOS ---
ARQ = "pns2019_cachorro_gato.csv"
CANDIDATOS = [ARQ, os.path.join("dados", ARQ), os.path.join("..", "dados", ARQ),
              os.path.join(os.path.expanduser("~"), "365Probabilidades", "dados", ARQ)]
caminho = next((c for c in CANDIDATOS if os.path.exists(c)), None)
if caminho is None:
    raise FileNotFoundError(f"Salve {ARQ} na mesma pasta deste notebook (pasta atual: {os.getcwd()}).")
d = pd.read_csv(caminho)

# Os quatro tipos de domicílio, por diferença entre as três tabelas do IBGE.
# total       = domicílios com cachorro / percentual com cachorro
# com_ou      = domicílios vacinados / percentual vacinado   (cachorro OU gato)
# ambos       = cachorro + gato - com_ou
d["total"] = d["cachorro_mil"] / (d["cachorro_pct"] / 100)
d["com_ou"] = d["vacinados_mil"] / (d["vacinados_pct"] / 100)
d["ambos"] = d["cachorro_mil"] + d["gato_mil"] - d["com_ou"]
d["so_cachorro"] = d["cachorro_mil"] - d["ambos"]
d["so_gato"] = d["gato_mil"] - d["ambos"]
d["nenhum"] = d["total"] - d["com_ou"]
for c in ["ambos", "so_cachorro", "so_gato", "nenhum"]:
    d["p_" + c] = d[c] / d["total"]
assert ((d["ambos"] >= 0) & (d["ambos"] <= d[["cachorro_mil", "gato_mil"]].min(axis=1))).all()

br = d[d["sigla"] == "BR"].iloc[0]
urb = d[d["sigla"] == "URB"].iloc[0]
rur = d[d["sigla"] == "RUR"].iloc[0]
uf = d[d["nivel"] == "UF"].copy()

print("=" * 72)
print("  OS DADOS - PNS 2019, IBGE (SIDRA 4930, 4931, 4932)")
print("=" * 72)
print(f"  Domicílios no Brasil (recuperado):      {mil(br['total'] * 1000)}")
print(f"  Com algum cachorro:                     {mil(br['cachorro_mil'] * 1000)}  {vir(br['cachorro_pct'])}% [{vir(br['cachorro_ic_inf'])}; {vir(br['cachorro_ic_sup'])}]")
print(f"  Com algum gato:                         {mil(br['gato_mil'] * 1000)}  {vir(br['gato_pct'])}% [{vir(br['gato_ic_inf'])}; {vir(br['gato_ic_sup'])}]")
print(f"  Com cachorro ou gato (recuperado):      {mil(br['com_ou'] * 1000)}  {pct(br['com_ou'] / br['total'])}")
print(f"  Publicação do IBGE: 39,4 milhões com cachorro ou gato, 72,0% com todos vacinados")
print()
print("  Os quatro tipos de casa (Brasil):")
for nome, c in [("só cachorro", "so_cachorro"), ("só gato", "so_gato"), ("os dois", "ambos"), ("nenhum", "nenhum")]:
    print(f"    {nome:<14} {mil(br[c] * 1000):>12}   {pct(br['p_' + c]):>6}")
p_indep = br["cachorro_pct"] / 100 * br["gato_pct"] / 100
OR = (br["ambos"] * br["nenhum"]) / (br["so_cachorro"] * br["so_gato"])
print(f"\n  Se cachorro e gato fossem independentes, 'os dois' seria {pct(p_indep)}; observado {pct(br['p_ambos'])}")
print(f"  Razão de chances cachorro × gato na mesma casa: {vir(OR, 2)}")
print()
print("  Urbano e rural:")
for r in (urb, rur):
    print(f"    {r['nome']:<8} só cachorro {pct(r['p_so_cachorro']):>6}  só gato {pct(r['p_so_gato']):>6}  os dois {pct(r['p_ambos']):>6}  nenhum {pct(r['p_nenhum']):>6}")
print()
print(f"  Estados: cachorro de {vir(uf['cachorro_pct'].min())}% ({uf.loc[uf['cachorro_pct'].idxmin(), 'nome']}) a {vir(uf['cachorro_pct'].max())}% ({uf.loc[uf['cachorro_pct'].idxmax(), 'nome']})")
print(f"           gato de {vir(uf['gato_pct'].min())}% ({uf.loc[uf['gato_pct'].idxmin(), 'nome']}) a {vir(uf['gato_pct'].max())}% ({uf.loc[uf['gato_pct'].idxmax(), 'nome']})")
print("=" * 72)


In [ ]:
# --- O MODELO ---
N_RUA = 10          # régua: uma rua de dez casas
N_SIM = 200_000

def p_rua(n, p_cao, p_gato):
    # probabilidade exata, pela multinomial, de uma rua de n casas ter
    # mais casas só de cachorro do que só de gato (as casas com os dois se anulam)
    # devolve (mais cachorros, empate, mais gatos)
    p_resto = 1 - p_cao - p_gato
    mais = empate = 0.0
    for a in range(n + 1):
        for b in range(n + 1 - a):
            c = n - a - b
            logm = lgamma(n + 1) - lgamma(a + 1) - lgamma(b + 1) - lgamma(c + 1)
            m = exp(logm + a * np.log(p_cao) + b * np.log(p_gato) + c * np.log(p_resto))
            if a > b:
                mais += m
            elif a == b:
                empate += m
    return mais, empate, 1 - mais - empate

# Camada 1: a rua de dez casas, Brasil
P_mais, P_emp, P_menos = p_rua(N_RUA, br["p_so_cachorro"], br["p_so_gato"])

# Camada 2: Monte Carlo como conferência (200.000 ruas, semente 42)
probs = [br["p_so_cachorro"], br["p_so_gato"], br["p_ambos"], br["p_nenhum"]]
ruas = ger.multinomial(N_RUA, probs, size=N_SIM)
casas_cao = ruas[:, 0] + ruas[:, 2]
casas_gato = ruas[:, 1] + ruas[:, 2]
mc_mais = np.mean(casas_cao > casas_gato)
mc_emp = np.mean(casas_cao == casas_gato)
dif = casas_cao - casas_gato

# Camada 3: a incerteza do IBGE carregada até a resposta
# cada quantidade publicada vira uma normal com desvio tirado do seu IC 95%
def se(ic_inf, ic_sup):
    return (ic_sup - ic_inf) / 3.92

N_INC = 20_000
s_cao = ger.normal(br["cachorro_pct"], se(br["cachorro_ic_inf"], br["cachorro_ic_sup"]), N_INC) / 100
s_gato = ger.normal(br["gato_pct"], se(br["gato_ic_inf"], br["gato_ic_sup"]), N_INC) / 100
s_vac_mil = ger.normal(br["vacinados_mil"], se(27989.786, 28817.320), N_INC)
s_vac_pct = ger.normal(br["vacinados_pct"], se(br["vacinados_ic_inf"], br["vacinados_ic_sup"]), N_INC) / 100
s_total = br["cachorro_mil"] / s_cao
s_ou = s_vac_mil / s_vac_pct
s_ambos = (br["cachorro_mil"] + br["gato_mil"] * (s_gato / (br["gato_pct"] / 100)) - s_ou) / s_total
s_pcao = s_cao - s_ambos
s_pgato = s_gato - s_ambos
# a conta exata é cara dentro de um laço de 20.000; usa-se a aproximação normal
# da diferença (só cachorro − só gato) em dez casas, conferida contra a exata no ponto central
def p_normal(n, pc, pg):
    mu = n * (pc - pg)
    var = n * (pc * (1 - pc) + pg * (1 - pg) + 2 * pc * pg)
    return 1 - stats.norm.cdf(0.5, mu, np.sqrt(var))
inc = p_normal(N_RUA, s_pcao, s_pgato)
inc_ic = np.percentile(inc, [2.5, 97.5])
desloc = p_normal(N_RUA, br["p_so_cachorro"], br["p_so_gato"]) - P_mais   # viés da aproximação no ponto central
inc_ic_corr = inc_ic - desloc

# Camada 4: o tamanho da rua
ns = np.arange(1, 41)
curva = np.array([p_rua(int(n), br["p_so_cachorro"], br["p_so_gato"]) for n in ns])

# Camada 5: urbano, rural e estados
P_urb = p_rua(N_RUA, urb["p_so_cachorro"], urb["p_so_gato"])[0]
P_rur = p_rua(N_RUA, rur["p_so_cachorro"], rur["p_so_gato"])[0]
uf["P10"] = [p_rua(N_RUA, a, b)[0] for a, b in zip(uf["p_so_cachorro"], uf["p_so_gato"])]
uf = uf.sort_values("P10", ascending=False)

# Camada 6: e se cachorro e gato fossem independentes na mesma casa?
P_indep = p_rua(N_RUA, br["cachorro_pct"] / 100 * (1 - br["gato_pct"] / 100),
                br["gato_pct"] / 100 * (1 - br["cachorro_pct"] / 100))[0]

print("=" * 72)
print("  O MODELO - A RUA DE DEZ CASAS")
print("=" * 72)
print(f"  Casas esperadas com cachorro: {vir(N_RUA * br['cachorro_pct'] / 100)}   com gato: {vir(N_RUA * br['gato_pct'] / 100)}")
print(f"  P(mais casas com cachorro do que com gato): {pct(P_mais)}")
print(f"  P(empate):                                  {pct(P_emp)}")
print(f"  P(mais casas com gato):                     {pct(P_menos)}")
print(f"  Monte Carlo, {mil(N_SIM)} ruas: {pct(mc_mais)} e {pct(mc_emp)} de empate (confere)")
print(f"  Diferença típica (casas com cachorro − casas com gato): mediana {int(np.median(dif))}, 80% entre {int(np.percentile(dif, 10))} e {int(np.percentile(dif, 90))}")
print(f"  Incerteza do IBGE carregada: {pct(inc_ic_corr[0])} a {pct(inc_ic_corr[1])}")
print()
print(f"  Se os bichos fossem independentes na casa: {pct(P_indep)} (quase não muda: 'os dois' se anulam)")
print(f"  Urbano: {pct(P_urb)}   Rural: {pct(P_rur)}")
print()
print("  O tamanho da rua:")
for n in [1, 2, 3, 5, 10, 20, 30, 40]:
    m, e, _ = curva[n - 1]
    print(f"    {n:>2} casas  mais cachorros {pct(m):>6}   empate {pct(e):>6}")
print()
print("  Estados, rua de dez casas:")
for _, r in uf.iterrows():
    print(f"    {r['sigla']}  {pct(r['P10']):>6}   só cachorro {pct(r['p_so_cachorro']):>6}  só gato {pct(r['p_so_gato']):>6}  os dois {pct(r['p_ambos']):>6}")
print("=" * 72)


In [ ]:
# --- VISUALIZAÇÃO ---

# ══ GRÁFICO 1 · a casa típica: os quatro tipos ══
fig1, ax1 = plt.subplots(figsize=(11, 5.6))
tipos = ["só cachorro", "os dois", "só gato", "nenhum dos dois"]
vals = [br["p_so_cachorro"], br["p_ambos"], br["p_so_gato"], br["p_nenhum"]]
cores = [VERMELHO, DOURADO, VERDE, CINZA]
esq = 0
for t, v, c in zip(tipos, vals, cores):
    ax1.barh([0], [v * 100], left=esq, color=c, alpha=0.9, height=0.55)
    ax1.text(esq + v * 50, 0, f"{t}\n{pct(v)}", ha="center", va="center", fontsize=11.5, color="white", fontweight="bold")
    esq += v * 100
ax1.set_xlim(0, 100); ax1.set_yticks([]); ax1.set_xlabel("domicílios do Brasil (%)")
ax1.grid(False)
ax1.annotate(f"casas com cachorro: {vir(br['cachorro_pct'])}%", xy=((vals[0] + vals[1]) * 50, 0.42), xytext=((vals[0] + vals[1]) * 50, 0.78),
             ha="center", fontsize=11, color=VERMELHO, arrowprops=dict(arrowstyle="-[, widthB=15.3, lengthB=0.6", color=VERMELHO, lw=1.4))
ax1.annotate(f"casas com gato: {vir(br['gato_pct'])}%", xy=((vals[0] + (vals[1] + vals[2]) / 2) * 100, -0.42), xytext=((vals[0] + (vals[1] + vals[2]) / 2) * 100, -0.8),
             ha="center", fontsize=11, color=VERDE, arrowprops=dict(arrowstyle="-[, widthB=6.4, lengthB=0.6", color=VERDE, lw=1.4))
ax1.set_ylim(-1, 1)
ax1.set_title("Em cada cem casas, 46 têm cachorro, 19 têm gato, 12 têm os dois, 46 não têm nenhum\n"
              "os quatro tipos de domicílio, Brasil 2019; 'os dois' sai por diferença entre três tabelas do IBGE", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Fonte: IBGE, PNS 2019, tabelas SIDRA 4930, 4931 e 4932  |  #365Probabilidades", ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-125-grafico-01-a-casa-tipica.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · a rua de dez casas: a diferença ══
fig2, ax2 = plt.subplots(figsize=(11, 6))
vals_d, cont = np.unique(dif, return_counts=True)
freq = cont / N_SIM * 100
cores2 = [VERMELHO if v > 0 else (DOURADO if v == 0 else VERDE) for v in vals_d]
ax2.bar(vals_d, freq, color=cores2, alpha=0.9, width=0.8)
for v, f in zip(vals_d, freq):
    if f >= 0.8 and v != 0:
        ax2.text(v, f + 0.4, vir(f) + "%", ha="center", fontsize=9.5, color="#333333")
ax2.set_xlabel("casas com cachorro menos casas com gato, numa rua de dez")
ax2.set_ylabel("ruas (%)")
ax2.set_xticks(range(-4, 10)); ax2.set_xlim(-4.7, 9.7)
ax2.text(5.2, max(freq) * 0.92, f"mais cachorros\n{pct(P_mais)} das ruas", color=VERMELHO, fontsize=12, fontweight="bold", ha="center")
ax2.text(-2.6, max(freq) * 0.92, f"mais gatos\n{pct(P_menos)}", color=VERDE, fontsize=11, fontweight="bold", ha="center")
ax2.text(0, freq[vals_d == 0][0] + 0.5, f"empate\n{pct(P_emp)}", color="#8a6f1f", fontsize=11, fontweight="bold", ha="center")
ax2.set_title(f"Numa rua de dez casas, o cachorro vence em {pct(P_mais)} das ruas; o gato, em {pct(P_menos)}\n"
              f"{mil(N_SIM)} ruas simuladas com as proporções do IBGE; a conta exata pela multinomial dá o mesmo", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Modelo próprio: multinomial sobre os quatro tipos de casa, PNS 2019 · semente 42  |  #365Probabilidades",
            ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-125-grafico-02-a-rua.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · o tamanho da rua ══
fig3, ax3 = plt.subplots(figsize=(11, 6))
ax3.plot(ns, curva[:, 0] * 100, color=VERMELHO, lw=2.8, marker="o", ms=4, label="mais casas com cachorro")
ax3.plot(ns, curva[:, 1] * 100, color=DOURADO, lw=2.2, marker="o", ms=4, label="empate")
ax3.plot(ns, curva[:, 2] * 100, color=VERDE, lw=2.2, marker="o", ms=4, label="mais casas com gato")
ax3.axvline(N_RUA, color="#333333", lw=1, ls="--")
ax3.text(N_RUA + 0.4, 50, f"dez casas\n{pct(P_mais)}", fontsize=11, color="#333333")
for n in [2, 5, 20, 40]:
    ax3.text(n + 0.6, curva[n - 1, 0] * 100 - 4.5, pct(curva[n - 1, 0]), ha="left", fontsize=10, color=VERMELHO)
ax3.set_xlabel("casas na rua"); ax3.set_ylabel("probabilidade (%)")
ax3.set_xlim(0, 43); ax3.set_ylim(0, 102)
ax3.legend(fontsize=10, loc="center right")
ax3.set_title("Com duas casas a disputa é quase moeda; com vinte, o cachorro vence em 97% das ruas\n"
              "a conta exata pela multinomial para cada tamanho de rua", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Modelo próprio sobre a PNS 2019: casas só de cachorro contra casas só de gato  |  #365Probabilidades",
            ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-125-grafico-03-o-tamanho-da-rua.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 3 salvo!")

# ══ GRÁFICO 4 · estado a estado ══
fig4, ax4 = plt.subplots(figsize=(11, 8.2))
u = uf.sort_values("P10")
y = np.arange(len(u))
cores4 = [VERMELHO if p >= P_mais else VERDE for p in u["P10"]]
ax4.barh(y, u["P10"] * 100, color=cores4, alpha=0.9, height=0.7)
for i, (_, r) in enumerate(u.iterrows()):
    ax4.text(r["P10"] * 100 + 0.6, i, pct(r["P10"]), va="center", fontsize=9.5, color="#333333",
             bbox=dict(facecolor="white", edgecolor="none", pad=0.6), zorder=5)
ax4.axvline(P_mais * 100, color="#333333", lw=1.2, ls="--", zorder=1)
ax4.text(P_mais * 100 - 0.8, len(u) - 0.6, f"Brasil {pct(P_mais)}", ha="right", fontsize=10.5, color="#333333")
ax4.set_yticks(y); ax4.set_yticklabels(u["nome"], fontsize=10)
ax4.set_xlim(0, 104); ax4.set_xlabel("chance de a rua de dez casas ter mais casas com cachorro do que com gato (%)")
ax4.set_title("Em Goiás o cachorro vence em 98% das ruas; em Alagoas e no Ceará, em 59%\n"
              "no Nordeste o gato chega a uma casa em cada quatro; no Centro-Oeste e no Sul o cachorro está em seis de cada dez", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Fonte: IBGE, PNS 2019 por unidade da federação · mesma conta, com os quatro tipos de cada estado  |  #365Probabilidades",
            ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-125-grafico-04-estado-a-estado.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 4 salvo!")


### 💡 O Insight

**Numa rua de dez casas, a chance de haver mais casas com cachorro do que com gato é de
87,9%.** Empata em 7,7%. O gato vence em 4,5%.

A sensação está certa. E o tamanho dela é este: em cada cem casas do Brasil, 46 têm
cachorro e 19 têm gato. Doze têm os dois. E 46, o mesmo número das casas com cachorro,
não têm nenhum.

A conta tem um atalho que vale guardar. As casas com os dois bichos não decidem nada,
porque contam um ponto para cada lado. A disputa é entre as 35 casas só de cachorro e as
8 casas só de gato. É por isso que a resposta quase não muda entre cidade e campo: no
campo há muito mais casa com os dois (29% contra 9%), mas a casa só de cachorro e a casa
só de gato têm a mesma frequência nos dois lugares.

O que muda a resposta é o mapa. Em Goiás, Paraná e Mato Grosso o cachorro vence em mais
de 97% das ruas. No Ceará, em Alagoas e no Maranhão, onde uma casa em cada quatro tem
gato, a disputa cai para perto de 60%.

E um detalhe que o IBGE não publicou de forma direta: quem tem cachorro tem o dobro de
chance de também ter gato. Os bichos não se excluem. Se acumulam.

*Na sua rua, você sabe quantas casas têm gato?*

---

### ⚠️ Limitações do Modelo

- A rua é uma hipótese: dez casas sorteadas da população brasileira. Casas da mesma rua
  se parecem (renda, quintal, campo ou cidade), e isso puxa a resposta para os dois
  extremos. A conta por estado e por situação urbana ou rural é a forma de mostrar
  quanto a vizinhança importa.
- O IBGE conta casas com cachorro e casas com gato, não quantos animais há em cada casa.
  A pergunta do dia é sobre casas; sobre cabeças, não há dado nas tabelas usadas.
- O número de casas com os dois bichos sai por diferença entre três tabelas, e uma delas
  (a vacinação) publica o percentual com uma casa decimal. O arredondamento move a
  estimativa de "os dois" em menos de um décimo de ponto; a propagação dos intervalos de
  confiança está na camada 3 do modelo.
- A PNS é de 2019. A publicação anterior (2013) mostrava 44,3% de casas com cachorro e
  17,7% com gato; as duas proporções subiram, e podem ter subido mais desde então.
- Os intervalos de confiança do IBGE já incorporam o desenho da amostra. Não se aplica
  o ×0,80, porque o dado é um fato da casa e não autorrelato de atitude; a convenção do
  projeto vale para proporções de survey sobre o que a pessoa diz de si.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
